In [ ]:
#Scores is an np.array, getting numbers, and these numbers can be any real number with no condition. Then using Gibbs measure, we can convert them to be probability distribution. We simply take each element of the array and exp of that element, devided by the normalizer(meaning sum of all exps) will give us a Gibbs probability measure.



import numpy as np
import matplotlib.pyplot as plt
#softmax is the last part of the algorithm in "Attention is all you need" paper. We here feed scores array to this softmax,but how that scores array produced in first place? That's what Attention does./ tau is the temprature in Gibbs definition. 
def softmax(scores, tau) : 
    scores = np.asarray(scores,dtype = float)
    z = np.max(scores)
    scores = scores - z
    exp_scores = np.exp(scores/tau)
    normalizer = np.sum(exp_scores)
    probabilities  = exp_scores/normalizer #Gibbs measure 
    L = -np.log(probabilities[1]) #Loss Function

    return probabilities, L

In [14]:
for tau in [0.1,1,10,100]:
    print(softmax(np.array([2,1,-1]), tau))

(array([9.99954602e-01, 4.53978687e-05, 9.35719815e-14]), np.float64(10.00004539889931))
(array([0.70538451, 0.25949646, 0.03511903]), np.float64(1.3490122167681864))
(array([0.37797814, 0.34200877, 0.28001309]), np.float64(1.0729189131256587))
(array([0.33778131, 0.33442033, 0.32779836]), np.float64(1.0953566081506636))


In [ ]:
#Here we will use the formulas in Attention paper, to calculate scores, then probability distribution, and then output using these formulas: s_j = \frac{q^T k_j}{sqrt{d_k}}, d_k is the number of components, so for our example here it's just 2. q is a vector, k is a vector, q^T gives another vector and q^T k_j matrix will give a scalar. 
#Then we will use th eformula alpha_j = e^{s_j}/sum e^{s_i}, so these alpha_j are Gibbs distribution, and finally output vector is o = sum alpha_jv_j. 

q = np.array([1,0])
k1 = np.array([2,0])
k2 = np.array([1,1])
k3 = np.array([-1,2])
v1 = np.array([10,0])
v2 = np.array([0,10])
v3 = np.array([4,4])

scores = []
d = len(q)

s1 = q.T @ k1 / np.sqrt(d)
s2 = q.T @ k2 / np.sqrt(d)
s3 = q.T @ k3 / np.sqrt(d)
scores.append(s1)
scores.append(s2)
scores.append(s3)

print(scores)


#Here we need to pass these scores through our softmax function, to get values for alpha_j.

alpha = softmax(scores,1)[0] 
print(alpha)

#Then here we calculate the output vector. 

output_vector = alpha[0] * v1 + alpha[1] * v2 + alpha[2] * v3
print(output_vector)


    



[np.float64(1.414213562373095), np.float64(0.7071067811865475), np.float64(-0.7071067811865475)]
[0.61998512 0.30569525 0.07431963]
[62.29579033  3.35423103]


In [31]:
# Here we will run the same experiment but this time, those vectors all stacked together to obtain a matrix. 
K = np.stack([k1,k2,k3])
V = np.stack([v1,v2,v3])

output = softmax((q @ K.T) / np.sqrt(d),1)[0] @ V
print(output)


[62.29579033  3.35423103]


In [20]:
#Multiple queries and Causal masking

import numpy as np
def softmax_rows(scores): 
    max_vector = np.max(scores, axis = 1, keepdims=True)
    scores = scores - max_vector
    exp_scores = np.exp(scores)
    sum_matrix = np.sum(exp_scores,axis = 1, keepdims=True)
    return exp_scores / sum_matrix

In [5]:
print(softmax_rows([[2,1,-1],[1002, 1001, 999]]))

[[0.70538451 0.25949646 0.03511903]
 [0.70538451 0.25949646 0.03511903]]


In [12]:
K = np.array([[2,0],[1,1],[-1,2]])
V = np.array([[10,0],[0,10],[4,4]])
q1 = (1,0)
q2 = (0,1)

Q = np.stack([q1,q2])
dk = 2

S = (Q @ K.T) / np.sqrt(dk)
A = softmax_rows(S)
O = A @ V

print(S)
print(A)
print(O)

[[ 1.41421356  0.70710678 -0.70710678]
 [ 0.          0.70710678  1.41421356]]
[[0.61998512 0.30569525 0.07431963]
 [0.14002925 0.28399541 0.57597535]]
[[6.4971297  3.35423103]
 [3.70419383 5.14385548]]


In [15]:
W_Q = np.array([[1,0],[0,1],[1,0],[0,1]])
W_K = np.array([[1,0],[0,1],[0,1],[1,0]])
W_V = np.array([[1,0,0],[0,1,0],[0,0,1],[1,1,0]])
X = np.array([[1,0,1,0],[0,1,0,1],[1,1,0,0]])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

dk = 2

#query : What this position uses to compare itslef with potential sources.
#key: waht this position offers for other queries to compare against. 
#value: What information this position contributes when selected. 

#softmax S_{ij} compares the query from destination i with the key from source j. Thus :
#Row i: all the sources that destination i is considering.
#Column j: how source j scores with each destination. 

S = (Q @ K.T)/ np.sqrt(dk)
A = softmax_rows(S)
O = A @ V

print(S)
print(A)
print(O)



[[1.41421356 1.41421356 1.41421356]
 [1.41421356 1.41421356 1.41421356]
 [1.41421356 1.41421356 1.41421356]]
[[0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]
 [0.33333333 0.33333333 0.33333333]]
[[1.         1.         0.33333333]
 [1.         1.         0.33333333]
 [1.         1.         0.33333333]]


In [22]:
#Causal Masking


#allowed = 0, blocked = -infty


Mask  = np.array([[0,-np.inf,-np.inf],[0,0,-np.inf],[0,0,0]])
A = softmax_rows(S + Mask)
O = A @ V

print(A)
print(O)



[[1.         0.         0.        ]
 [0.5        0.5        0.        ]
 [0.05285739 0.05285739 0.89428521]]
[[1.         0.         1.        ]
 [1.         1.         0.5       ]
 [1.89428521 1.89428521 0.94714261]]


In [23]:
W_Q = np.array([[1,0],[0,1],[1,0],[0,1]])
W_K = np.array([[1,0],[0,1],[0,1],[1,0]])
W_V = np.array([[1,0,0],[0,1,0],[0,0,1],[1,1,0]])
X = np.array([[1,0,1,0],[0,1,0,1],[1,1,1,1]])



Q = X @ W_Q
K = X @ W_K
V = X @ W_V

dk = 2

#query : What this position uses to compare itslef with potential sources.
#key: waht this position offers for other queries to compare against. 
#value: What information this position contributes when selected. 

#softmax S_{ij} compares the query from destination i with the key from source j. Thus :
#Row i: all the sources that destination i is considering.
#Column j: how source j scores with each destination. 

S = (Q @ K.T)/ np.sqrt(dk)
A = softmax_rows(S + Mask)
O = A @ V

print(S)
print(A)
print(O)

[[1.41421356 1.41421356 2.82842712]
 [1.41421356 1.41421356 2.82842712]
 [2.82842712 2.82842712 5.65685425]]
[[1.         0.         0.        ]
 [0.5        0.5        0.        ]
 [0.05285739 0.05285739 0.89428521]]
[[1.         0.         1.        ]
 [1.         1.         0.5       ]
 [1.89428521 1.89428521 0.94714261]]


In [16]:
def attention(X,W_Q,W_K, W_V):
   Q = X @ W_Q #query
   K = X @ W_K #keys
   V = X @ W_V #value
   dk = Q.shape[1] #the number of columns is the query/ key dimension
   n = X.shape[0] #number of rows in X
   Mask = np.zeros((n,n), dtype = float)
   for i in range(n):
      for j in range(n):
         if j <= i:
            Mask[i][j] = 0
         else: 
            Mask[i][j] = -np.inf
   

   S = (Q @ K.T)/ np.sqrt(dk)
   A = softmax_rows(S + Mask)
   O = A @ V   
   return A, S, O


In [ ]:
# SINGLE-HEAD CAUSAL SELF-ATTENTION
#
# X contains one representation vector per token position:
#     X: (n, d_model)
#
# Shared projection matrices create queries, keys, and values:
#     Q = X @ W_Q     shape: (n, d_k)
#     K = X @ W_K     shape: (n, d_k)
#     V = X @ W_V     shape: (n, d_v)
#
# Each query identifies a destination; keys describe potential sources.
# Values contain the information those sources contribute.
#
# S[i, j] = dot(Q[i], K[j]) / sqrt(d_k)
# measures compatibility between destination i and source j.
#
# The causal mask leaves j <= i unchanged and blocks j > i with -inf.
# Apply the mask BEFORE row-wise softmax. Blocked weights become zero,
# and each row's allowed weights sum to one.
#
# A: (n, n)     attention weights
# O = A @ V     output shape: (n, d_v)
#
# O[i] is a weighted average of values from positions 0 through i.
# Position i may include itself because it helps predict token i+1.
#
# Tests completed:
# - Different sequence lengths work.
# - Every attention-weight row sums to approximately 1.
# - All weights above the diagonal are zero.
# - Changing the final input leaves earlier outputs unchanged.
# - The notebook runs successfully after restarting the kernel.
#
# Scope: this is one attention head with manually chosen projections.
# It is not yet a trained Transformer or a next-token predictor.

3


In [35]:
import numpy as np
vocabulary = { 
    "I": 0,
    "love": 1,
    "math": 2,
    ".": 3
}
#dictionary in python

print(vocabulary["love"])
sentence = "I love love math ."
tokens = sentence.split()
token_ids = []
for token in tokens:
    token_ids.append(vocabulary[token])


print(token_ids)


E = np.array([[1,0,1,0],
              [0,1,0,1],
              [1,1,0,0],
              [0,0,1,1]])
X = E[token_ids]
print(X)


final = attention(X,W_Q,W_K, W_V)
print(final)

W_vocab = np.array([[1,0,-1,0],
                    [0,1,1,-1],
                    [-1,0,0,1]])

Z = attention(X,W_Q,W_K, W_V)[2] @ W_vocab
P = softmax_rows(Z)

print(P)
print(P.shape)

print(np.sum(P,axis = 1))
    
    



1
[0, 1, 1, 2, 3]
[[1 0 1 0]
 [0 1 0 1]
 [0 1 0 1]
 [1 1 0 0]
 [0 0 1 1]]
(array([[1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.5       , 0.5       , 0.        , 0.        , 0.        ],
       [0.33333333, 0.33333333, 0.33333333, 0.        , 0.        ],
       [0.25      , 0.25      , 0.25      , 0.25      , 0.        ],
       [0.2       , 0.2       , 0.2       , 0.2       , 0.2       ]]), array([[1.41421356, 1.41421356, 1.41421356, 1.41421356, 1.41421356],
       [1.41421356, 1.41421356, 1.41421356, 1.41421356, 1.41421356],
       [1.41421356, 1.41421356, 1.41421356, 1.41421356, 1.41421356],
       [1.41421356, 1.41421356, 1.41421356, 1.41421356, 1.41421356],
       [1.41421356, 1.41421356, 1.41421356, 1.41421356, 1.41421356]]), array([[1.        , 0.        , 1.        ],
       [1.        , 1.        , 0.5       ],
       [1.        , 1.33333333, 0.33333333],
       [1.        , 1.25      , 0.25      ],
       [1.        , 1.2       , 0.4       ]]))
[[0.

SyntaxError: invalid syntax (2965680911.py, line 2)